# Module 26 — TF-IDF & BM25: Building a Search Engine from Scratch

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Intermediate  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Scikit-learn, Rank-BM25, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 25 (Text Preprocessing)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Search Corpus](#3-synthetic-search-corpus)
4. [TF-IDF Implementation](#4-tfidf-implementation)
5. [BM25 Implementation](#5-bm25-implementation)
6. [Search Evaluation](#6-search-evaluation)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why TF-IDF and BM25 for search?

TF-IDF and BM25 are **foundational** search algorithms:
- **TF-IDF**: Term Frequency-Inverse Document Frequency.
- **BM25**: Best Matching 25, industry-standard ranking function.
- **Bag-of-words**: Simple yet effective for many search tasks.

**Applications at Yandex**:
1. **Web search**: Rank web pages by relevance to query.
2. **Product search**: Match queries to product listings.
3. **Document retrieval**: Find relevant documents in knowledge base.

**Key concepts**:
- **Term frequency (TF)**: How often a term appears in a document.
- **Inverse document frequency (IDF)**: How rare a term is across corpus.
- **BM25 scoring**: Advanced TF saturation and length normalization.

In this notebook we will:
1. Generate a synthetic product corpus.
2. Implement TF-IDF search.
3. Implement BM25 search.
4. Evaluate with NDCG@10.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Search libraries ─────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

# ── NLP ──────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Search Corpus

In [ ]:
# Generate synthetic product corpus
N_PRODUCTS = 10000

# Product templates
categories = {
    'electronics': ['laptop', 'smartphone', 'tablet', 'headphones', 'camera'],
    'clothing': ['t-shirt', 'jeans', 'dress', 'jacket', 'shoes'],
    'books': ['novel', 'textbook', 'cookbook', 'biography', 'sci-fi'],
    'home': ['furniture', 'kitchen', 'decor', 'bedding', 'lighting'],
    'sports': ['fitness', 'outdoor', 'team sports', 'water sports', 'winter sports']
}

adjectives = ['premium', 'affordable', 'luxury', 'professional', 'high-quality', 'modern', 'classic']
features = ['wireless', 'portable', 'durable', 'lightweight', 'waterproof', 'smart', 'ergonomic']

# Generate products
products = []
for i in range(N_PRODUCTS):
    category = np.random.choice(list(categories.keys()))
    product_type = np.random.choice(categories[category])
    adj = np.random.choice(adjectives)
    feat1 = np.random.choice(features)
    feat2 = np.random.choice(features)
    
    title = f"{adj} {feat1} {product_type}"
    description = f"This {adj} {product_type} features {feat1} and {feat2} design. Perfect for {category} enthusiasts."
    
    products.append({
        'product_id': i,
        'category': category,
        'title': title,
        'description': description,
        'full_text': f"{title} {description}"
    })

df_products = pd.DataFrame(products)

print(f"✓ Generated {N_PRODUCTS} products")
print(f"\nCategories:")
print(df_products['category'].value_counts())
print(f"\nSample products:")
for i in range(3):
    print(f"  {i+1}. {df_products['title'].iloc[i]}")

---
## §4 · TF-IDF Implementation

In [ ]:
# Preprocess text for search
stop_words = set(stopwords.words('english'))

def preprocess(text):
    """Preprocess text for search."""
    tokens = word_tokenize(text.lower())
    return ' '.join([t for t in tokens if t.isalpha() and t not in stop_words])

df_products['processed'] = df_products['full_text'].apply(preprocess)

# Build TF-IDF index
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df_products['processed'])

print(f"✓ TF-IDF index built")
print(f"  Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"  Matrix shape: {tfidf_matrix.shape}")

# Search function
def tfidf_search(query, top_k=10):
    """Search using TF-IDF cosine similarity."""
    query_processed = preprocess(query)
    query_vec = tfidf_vectorizer.transform([query_processed])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = scores.argsort()[-top_k:][::-1]
    return [(idx, scores[idx]) for idx in top_indices]

# Test search
query = "wireless headphones"
results = tfidf_search(query)
print(f"\nTF-IDF results for '{query}':")
for idx, score in results[:5]:
    print(f"  Score: {score:.4f} - {df_products['title'].iloc[idx]}")

---
## §5 · BM25 Implementation

In [ ]:
# Build BM25 index
tokenized_corpus = [doc.split() for doc in df_products['processed']]
bm25 = BM25Okapi(tokenized_corpus)

print(f"✓ BM25 index built")
print(f"  Corpus size: {len(tokenized_corpus)} documents")

# Search function
def bm25_search(query, top_k=10):
    """Search using BM25."""
    query_tokens = preprocess(query).split()
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[-top_k:][::-1]
    return [(idx, scores[idx]) for idx in top_indices]

# Test search
results = bm25_search(query)
print(f"\nBM25 results for '{query}':")
for idx, score in results[:5]:
    print(f"  Score: {score:.4f} - {df_products['title'].iloc[idx]}")

---
## §6 · Search Evaluation

In [ ]:
# Evaluate search quality
# Create test queries with known relevant documents
test_queries = [
    {'query': 'wireless headphones', 'relevant_category': 'electronics'},
    {'query': 'premium laptop', 'relevant_category': 'electronics'},
    {'query': 'affordable t-shirt', 'relevant_category': 'clothing'},
    {'query': 'luxury furniture', 'relevant_category': 'home'},
    {'query': 'professional camera', 'relevant_category': 'electronics'},
]

def calculate_ndcg(retrieved_indices, relevant_category, k=10):
    """Calculate NDCG@k."""
    dcg = 0
    for i, idx in enumerate(retrieved_indices[:k]):
        if df_products['category'].iloc[idx] == relevant_category:
            dcg += 1 / np.log2(i + 2)
    
    # Ideal DCG
    ideal_dcg = sum(1 / np.log2(i + 2) for i in range(min(k, 10)))
    
    return dcg / ideal_dcg if ideal_dcg > 0 else 0

# Evaluate both methods
tfidf_ndcgs = []
bm25_ndcgs = []

for test in test_queries:
    # TF-IDF
    tfidf_results = tfidf_search(test['query'], top_k=10)
    tfidf_indices = [idx for idx, _ in tfidf_results]
    tfidf_ndcgs.append(calculate_ndcg(tfidf_indices, test['relevant_category']))
    
    # BM25
    bm25_results = bm25_search(test['query'], top_k=10)
    bm25_indices = [idx for idx, _ in bm25_results]
    bm25_ndcgs.append(calculate_ndcg(bm25_indices, test['relevant_category']))

print("=" * 70)
print("SEARCH EVALUATION RESULTS")
print("=" * 70)
print(f"\nTF-IDF NDCG@10: {np.mean(tfidf_ndcgs):.4f}")
print(f"BM25 NDCG@10:   {np.mean(bm25_ndcgs):.4f}")
print(f"\n💡 BM25 typically outperforms TF-IDF due to better term saturation")

---
## §7 · Visualization

In [ ]:
# Compare TF-IDF and BM25 scores
fig = make_subplots(rows=1, cols=2, subplot_titles=['Score Distribution', 'NDCG@10 Comparison'])

# Score distribution
query = "wireless headphones"
tfidf_results = tfidf_search(query, top_k=100)
bm25_results = bm25_search(query, top_k=100)

fig.add_trace(go.Histogram(
    x=[score for _, score in tfidf_results],
    name='TF-IDF',
    opacity=0.7,
    nbinsx=30
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=[score for _, score in bm25_results],
    name='BM25',
    opacity=0.7,
    nbinsx=30
), row=1, col=1)

# NDCG comparison
fig.add_trace(go.Bar(
    x=['TF-IDF', 'BM25'],
    y=[np.mean(tfidf_ndcgs), np.mean(bm25_ndcgs)],
    marker_color=['#636EFA', '#EF553B']
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - BM25 provides better score distribution")
print("   - BM25 handles term saturation better")
print("   - Both methods are fast enough for real-time search")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Search Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Product search, document retrieval, knowledge base |
> | **Algorithm** | BM25 > TF-IDF for most search tasks |
> | **Evaluation** | NDCG@10, MRR, Precision@K |
> | **Indexing** | Pre-compute and store in inverted index |
> | **Latency** | < 50ms for real-time search |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Search pipeline**:
>    ```
>    Query → Preprocess → BM25 retrieval → Re-ranking → Results
>    ```
>
> 2. **Hybrid approaches**:
>    | Stage | Method | Purpose |
>    |-------|--------|---------|
>    | Retrieval | BM25 | Fast candidate generation |
>    | Re-ranking | BERT | Semantic understanding |
>    | Blending | Ensemble | Combine signals |
>
> 3. **Production considerations**:
>    - Use inverted indexes for fast retrieval
>    - Cache frequent queries
>    - A/B test ranking changes
>
> ### 🔑 Key Takeaways
>
> 1. **BM25 is the industry standard** for lexical search.
> 2. **TF-IDF is simpler** but less effective.
> 3. **NDCG@10** is the standard metric for search quality.
> 4. **Hybrid search** (lexical + semantic) provides best results.
> 5. **Latency is critical** for real-time search applications.